In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("master_dataset.csv")
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["store_id", "product_id", "date"]
).reset_index(drop=True)

print(df.shape)
df.head()

(1795, 33)


,date,store_id,product_id,demand,selling_price,discount_pct,promotion_flag,opening,received,sold,...,avg_daily_customers,region,temp_c,rain_mm,holiday,festival,weekend,local_event,revenue,stockout_flag
0,2026-07-01,S01,P101,107.0,60.0,0.0,0,264,20,107,...,1250,West,30.1,0.0,0,0,0,0,6420.0,0
1,2026-07-02,S01,P101,75.0,60.0,0.0,0,177,0,75,...,1250,West,30.2,8.6,0,0,0,0,4500.0,0
2,2026-07-03,S01,P101,93.0,60.0,0.0,0,102,80,93,...,1250,West,30.1,1.2,0,0,0,0,5580.0,0
3,2026-07-04,S01,P101,15.0,60.0,0.0,1,15,0,15,...,1250,West,30.6,0.0,0,0,1,0,900.0,1
4,2026-07-05,S01,P101,40.0,60.0,0.0,0,0,40,40,...,1250,West,31.8,4.1,0,0,1,0,2400.0,1


In [ ]:
df["day_of_week"] = df["date"].dt.dayofweek
df["weekend_flag"] = (df["day_of_week"] >= 5).astype(int)
df["month"] = df["date"].dt.month
df["week_no"] = df["date"].dt.isocalendar().week.astype(int)

In [ ]:
df["festival_flag"] = df["festival"].fillna(0).astype(int)

In [ ]:
group = df.groupby(
    ["store_id", "product_id"]
)["demand"]

df["lag_1"] = group.shift(1)
df["lag_7"] = group.shift(7)
df["lag_14"] = group.shift(14)

In [ ]:
df["rolling_mean_7"] = group.transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df["rolling_mean_14"] = group.transform(
    lambda x: x.shift(1).rolling(14).mean()
)

df["rolling_std_7"] = group.transform(
    lambda x: x.shift(1).rolling(7).std()
)

In [ ]:
df["days_of_inventory"] = (
    df["closing"] /
    df["rolling_mean_7"].replace(0, np.nan)
)

In [ ]:
df["inventory_to_demand_ratio"] = (
    df["closing"] /
    df["rolling_mean_7"].replace(0, np.nan)
)

In [ ]:
df["reorder_gap"] = (
    df["closing"] - df["reorder_lvl"]
)

In [ ]:
df = df.sort_values(
    ["store_id", "product_id", "date"]
)

df["price_change"] = (
    df.groupby(["store_id", "product_id"])["selling_price"]
    .pct_change()
)

In [ ]:
df["next_7_day_demand"] = (
    df.groupby(["store_id", "product_id"])["demand"]
    .transform(
        lambda x: x.shift(-1).rolling(7).sum()
    )
)

In [ ]:
df["next_7_day_demand"] = (
    df.groupby(["store_id", "product_id"])["demand"]
    .transform(
        lambda x: sum(x.shift(-i) for i in range(1, 8))
    )
)

In [ ]:
df["stockout_flag"] = (
    (df["closing"] == 0) &
    (df["demand"] > 0)
).astype(int)

In [ ]:
print(df["stockout_flag"].value_counts())

stockout_flag
0    1028
1     767
Name: count, dtype: int64


In [ ]:
model_df = df.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "rolling_mean_7",
        "rolling_mean_14",
        "next_7_day_demand"
    ]
).copy()

In [ ]:
print(model_df.shape)
print(model_df.isnull().sum())

(955, 49)
date                         0
store_id                     0
product_id                   0
demand                       0
selling_price                0
discount_pct                 0
promotion_flag               0
opening                      0
received                     0
sold                         0
closing                      0
reorder_lvl                  0
lead_days                    0
category                     0
sub_category                 0
brand                        0
mrp                          0
cost_price                   0
shelf_life_days              0
supplier_id                  0
city                         0
store_type                   0
floor_area_sqft              0
avg_daily_customers          0
region                       0
temp_c                       0
rain_mm                      0
holiday                      0
festival                     0
weekend                      0
local_event                  0
revenue                      

In [ ]:
model_df.to_csv(
    "model_dataset.csv",
    index=False
)